# Speech Emotion Classification: Two Pipeline Comparison

This notebook implements two approaches to classify emotions from speech:
1. **Audio-Direct Pipeline**: Wave2Vec embeddings → Emotion Classifier
2. **Text-Based Pipeline**: ASR (Whisper) → Text → Sentiment Classifier

Both pipelines use frozen foundation models with trainable classifiers on the RAVDESS dataset.

In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torchaudio
import librosa
from pathlib import Path
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import json

# Deep Learning and ML
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter

# Transformers and Models
import transformers
from transformers import AutoFeatureExtractor, AutoModel, pipeline
import whisper

# Metrics and utilities
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\malopif\miniconda3\envs\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: argument of type 'NoneType' is not iterable

In [ ]:
# Configuration
@dataclass
class Config:
    """Configuration for the emotion classification pipelines"""
    # Dataset
    ravdess_path: str = "./data/RAVDESS"  # Path to RAVDESS dataset
    sample_rate: int = 16000
    
    # Wave2Vec settings
    wave2vec_model: str = "facebook/wav2vec2-base"
    wave2vec_dim: int = 768
    
    # ASR + Sentiment settings
    whisper_model: str = "base"  # tiny, base, small, medium, large
    sentiment_model: str = "distilbert-base-uncased-finetuned-sst-2-english"
    
    # Classifier settings
    emotion_classes: List[str] = None
    num_emotions: int = 8  # RAVDESS has 8 emotions
    
    # Training
    batch_size: int = 16
    num_epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    
    # Paths
    output_dir: str = "./outputs"
    
    def __post_init__(self):
        if self.emotion_classes is None:
            # RAVDESS emotions: neutral, calm, happy, sad, angry, fearful, disgusted, surprised
            self.emotion_classes = [
                "neutral", "calm", "happy", "sad", 
                "angry", "fearful", "disgusted", "surprised"
            ]
        os.makedirs(self.output_dir, exist_ok=True)

config = Config()
print("Configuration:")
print(f"  Wave2Vec model: {config.wave2vec_model}")
print(f"  Whisper model size: {config.whisper_model}")
print(f"  Sentiment model: {config.sentiment_model}")
print(f"  Emotion classes: {config.emotion_classes}")

## RAVDESS Dataset Loading & Preprocessing

In [ ]:
def download_ravdess():
    """Download RAVDESS dataset (requires manual download or use zenodo API)"""
    import subprocess
    data_dir = Path(config.ravdess_path)
    
    if data_dir.exists():
        print(f"RAVDESS dataset already exists at {data_dir}")
        return
    
    print("\nRAVDESS dataset needs to be downloaded manually:")
    print("1. Go to: https://zenodo.org/record/1188976")
    print("2. Download all .zip files")
    print("3. Extract them to:", config.ravdess_path)
    print("\nAlternatively, you can download using wget/curl or the zenodo API")

class RAVDESSDataset(Dataset):
    """RAVDESS Speech Emotion Dataset"""
    
    def __init__(self, data_dir: str, split: str = "train", test_size: float = 0.2, val_size: float = 0.1):
        """
        Args:
            data_dir: Path to RAVDESS dataset
            split: 'train', 'val', or 'test'
            test_size: Proportion for test set
            val_size: Proportion for validation set
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.samples = []
        
        # RAVDESS emotion mapping (from filename)
        # Format: 03-02-01-01-02-01-12.wav
        # Position 3 (0-indexed: position 2) = emotion (1-8)
        emotion_map = {
            1: 0,  # neutral
            2: 1,  # calm
            3: 2,  # happy
            4: 3,  # sad
            5: 4,  # angry
            6: 5,  # fearful
            7: 6,  # disgusted
            8: 7   # surprised
        }
        
        # Collect all audio files
        audio_files = list(self.data_dir.glob("*.wav"))
        
        if not audio_files:
            print(f"No .wav files found in {self.data_dir}")
            print("Please download the RAVDESS dataset first")
            return
        
        # Parse emotion labels from filenames
        for audio_file in audio_files:
            name = audio_file.stem  # Remove .wav
            parts = name.split("-")
            
            if len(parts) >= 3:
                emotion_id = int(parts[2])  # Extract emotion code
                if emotion_id in emotion_map:
                    emotion_label = emotion_map[emotion_id]
                    self.samples.append({
                        'path': audio_file,
                        'emotion': emotion_label,
                        'emotion_name': config.emotion_classes[emotion_label]
                    })
        
        # Split data
        np.random.shuffle(self.samples)
        n_train = int(len(self.samples) * (1 - test_size - val_size))
        n_val = int(len(self.samples) * val_size)
        
        if split == "train":
            self.samples = self.samples[:n_train]
        elif split == "val":
            self.samples = self.samples[n_train:n_train+n_val]
        elif split == "test":
            self.samples = self.samples[n_train+n_val:]
        
        print(f"Loaded {len(self.samples)} samples for {split} split")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample['path']
        emotion = sample['emotion']
        
        # Load audio
        waveform, sr = torchaudio.load(str(path))
        
        # Resample if needed
        if sr != config.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, config.sample_rate)
            waveform = resampler(waveform)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # Squeeze to 1D
        waveform = waveform.squeeze(0)
        
        return {
            'waveform': waveform,
            'emotion': torch.tensor(emotion, dtype=torch.long),
            'path': str(path)
        }

# Try to download dataset
download_ravdess()

## Pipeline 1: Audio-Direct (Wave2Vec → Emotion Classifier)

In [ ]:
        return logits
    
    def get_trainable_params(self):
        """Get only the trainable parameters (classifier only)"""
        return self.classifier.parameters()

print("Audio-Direct Pipeline modules defined")

## Pipeline 2: Text-Based (ASR → Text Sentiment Classifier)

In [ ]:
        return logits, text
    
    def get_trainable_params(self):
        """Get only the trainable parameters (classifier only)"""
        return self.classifier.parameters()

print("Text-Based Pipeline modules defined")

## Training Utilities

In [ ]:
            'predictions': preds,
            'labels': labels
        }

print("Trainer class defined")

## Evaluation & Comparison

In [ ]:
    df = pd.DataFrame(comparison)
    print("\n" + "="*60)
    print("PIPELINE COMPARISON")
    print("="*60)
    print(df.to_string(index=False))
    print("="*60 + "\n")
    
    return df

print("Evaluation functions defined")

## Main Execution: Load Data & Train Models

In [ ]:
# Note: This requires the RAVDESS dataset to be downloaded
# If you don't have it yet, uncomment the download_ravdess() call above and download manually

print("Loading datasets...")

try:
    # Create datasets
    train_dataset = RAVDESSDataset(config.ravdess_path, split="train")
    val_dataset = RAVDESSDataset(config.ravdess_path, split="val")
    test_dataset = RAVDESSDataset(config.ravdess_path, split="test")
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
    
    print(f"Train set: {len(train_dataset)} samples")
    print(f"Val set: {len(val_dataset)} samples")
    print(f"Test set: {len(test_dataset)} samples")
    
    # Initialize models
    print("\nInitializing models...")
    audio_direct_model = AudioDirectPipeline(num_emotions=config.num_emotions).to(device)
    text_based_model = TextBasedPipeline(num_emotions=config.num_emotions).to(device)
    
    print("Audio-Direct Pipeline initialized")
    print("Text-Based Pipeline initialized")
    
    # Create trainers
    audio_trainer = Trainer(audio_direct_model, "Audio-Direct", device)
    text_trainer = Trainer(text_based_model, "Text-Based", device)
    
    # Train both pipelines
    trainers = {}
    
    print("\n" + "="*60)
    print("STARTING TRAINING")
    print("="*60)
    
    # Train Audio-Direct Pipeline
    audio_trainer.train(train_loader, val_loader, 
                       num_epochs=config.num_epochs, 
                       learning_rate=config.learning_rate)
    trainers["Audio-Direct"] = audio_trainer
    
    # Train Text-Based Pipeline
    # text_trainer.train(train_loader, val_loader, 
    #                    num_epochs=config.num_epochs, 
    #                    learning_rate=config.learning_rate)
    # trainers["Text-Based"] = text_trainer
    
    print("\nTraining completed!")
    
except FileNotFoundError as e:
    print(f"\nDataset not found: {e}")
    print("\nTo use this notebook:")
    print("1. Download RAVDESS dataset from: https://zenodo.org/record/1188976")
    print("2. Extract to:", config.ravdess_path)
    print("3. Rerun this cell")
    print("\nAlternatively, you can preprocess the data in a separate step.")

## Test & Compare Results

In [ ]:
# Test both pipelines on the test set
if 'trainers' in locals() and trainers:
    results = {}
    
    for model_name, trainer in trainers.items():
        result = trainer.test(test_loader)
        results[model_name] = result
    
    # Compare results
    comparison_df = compare_pipelines(results)
    
    # Plot training history
    plot_training_history(trainers, save_path=f"{config.output_dir}/training_history.png")
    
    # Plot confusion matrices
    plot_confusion_matrices(results, save_path=f"{config.output_dir}/confusion_matrices.png")
    
    print("\n✓ Evaluation completed!")
    print(f"✓ Results saved to {config.output_dir}/")
else:
    print("⚠️  No trained models found. Please run the training cell first.")

## Inference on New Audio

In [ ]:
def infer_emotion(audio_path: str, models: Dict[str, nn.Module]) -> Dict[str, dict]:
    """Infer emotions from new audio file using both pipelines"""
    
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    
    # Resample if needed
    if sr != config.sample_rate:
        resampler = torchaudio.transforms.Resample(sr, config.sample_rate)
        waveform = resampler(waveform)
    
    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0).to(device)
    
    results = {}
    
    # Audio-Direct Pipeline
    if "audio_direct_model" in models:
        model = models["audio_direct_model"]
        model.eval()
        with torch.no_grad():
            logits = model(waveform.unsqueeze(0))
        
        probs = torch.softmax(logits, dim=1)
        pred_emotion = config.emotion_classes[logits.argmax(dim=1).item()]
        pred_confidence = probs.max().item()
        
        results["Audio-Direct"] = {
            "emotion": pred_emotion,
            "confidence": pred_confidence,
            "probabilities": {
                config.emotion_classes[i]: probs[0, i].item() 
                for i in range(len(config.emotion_classes))
            }
        }
    
    # Text-Based Pipeline
    if "text_based_model" in models:
        model = models["text_based_model"]
        model.eval()
        with torch.no_grad():
            logits, transcribed_text = model(waveform)
        
        probs = torch.softmax(logits, dim=1)
        pred_emotion = config.emotion_classes[logits.argmax(dim=1).item()]
        pred_confidence = probs.max().item()
        
        results["Text-Based"] = {
            "emotion": pred_emotion,
            "confidence": pred_confidence,
            "transcribed_text": transcribed_text,
            "probabilities": {
                config.emotion_classes[i]: probs[0, i].item() 
                for i in range(len(config.emotion_classes))
            }
        }
    
    return results


def visualize_inference_results(results: Dict[str, dict]):
    """Visualize inference results from both pipelines"""
    
    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5))
    
    if len(results) == 1:
        axes = [axes]
    
    for idx, (model_name, result) in enumerate(results.items()):
        # Extract probabilities
        probs = result['probabilities']
        emotions = list(probs.keys())
        confidence_scores = list(probs.values())
        
        # Plot as bar chart
        colors = ['green' if emotions[i] == result['emotion'] else 'blue' 
                 for i in range(len(emotions))]
        axes[idx].barh(emotions, confidence_scores, color=colors)
        axes[idx].set_xlabel("Confidence")
        axes[idx].set_title(f"{model_name}\nPredicted: {result['emotion']}")
        axes[idx].set_xlim(0, 1)
        
        # Add transcribed text if available
        if 'transcribed_text' in result:
            axes[idx].text(0.5, -0.15, f"Transcribed: {result['transcribed_text']}", 
                          ha='center', transform=axes[idx].transAxes, fontsize=9)
    
    plt.tight_layout()
    plt.show()


# Example usage (uncomment and modify path when you have audio files)
# audio_path = "./data/test_audio.wav"
# inference_results = infer_emotion(audio_path, {
#     "audio_direct_model": audio_direct_model if 'audio_direct_model' in locals() else None,
#     "text_based_model": text_based_model if 'text_based_model' in locals() else None
# })
# 
# # Display results
# print("Inference Results:")
# for model_name, result in inference_results.items():
#     print(f"\n{model_name}:")
#     print(f"  Predicted Emotion: {result['emotion']}")
#     print(f"  Confidence: {result['confidence']:.4f}")
#     if 'transcribed_text' in result:
#         print(f"  Transcribed Text: {result['transcribed_text']}")
# 
# visualize_inference_results(inference_results)

print("✓ Inference utilities defined. Use infer_emotion() for new audio files.")

## Summary & Quick Reference

### Architecture Overview

**Pipeline 1: Audio-Direct (Wave2Vec)**
- Frozen Foundation: `wav2vec2-base` (768-dim embeddings)
- Trainable Classifier: 3-layer MLP (768 → 256 → 128 → 8 emotions)
- Input: Raw audio waveform
- Pros: Direct audio processing, minimal latency
- Cons: Less interpretability

**Pipeline 2: Text-Based (ASR + Sentiment)**
- Frozen ASR: `Whisper` (speech-to-text)
- Frozen Embedder: `distilbert-base-uncased-finetuned-sst-2-english`
- Trainable Classifier: 3-layer MLP (768 → 256 → 128 → 8 emotions)
- Input: Raw audio → transcribed text → embeddings
- Pros: Interpretable (can see transcribed text), leverages sentiment analysis
- Cons: Cascading errors from ASR, higher latency

### Key Features

✓ **Frozen Foundation Models**: Foundation models are frozen, only classifiers are trained
✓ **RAVDESS Dataset**: 1440 samples (8 emotions × 2 actors × 2 genders × × 60 conditions)
✓ **Flexible Training**: Easy to switch between models, adjust hyperparameters in Config
✓ **Comprehensive Evaluation**: Confusion matrices, F1-scores, detailed classification reports
✓ **Inference Functions**: Easy-to-use inference on new audio files

### Usage

1. **Download RAVDESS**: https://zenodo.org/record/1188976
2. **Configure path**: Update `config.ravdess_path` in Config
3. **Run training cells**: Execute data loading and training
4. **Evaluate**: Check test results and comparison plots
5. **Inference**: Use `infer_emotion()` on new audio files

### Customization

- Change model sizes in Config (e.g., `whisper_model = "large"` for better accuracy)
- Adjust batch size, learning rate, epochs
- Add data augmentation in RAVDESSDataset
- Extend classifier architectures in EmotionClassifier / TextEmotionClassifier
- Switch to different emotion datasets (TESS, EmoV-DB, etc.)